In [7]:
# Tìm thư mục gốc để notebook chạy được từ notebooks/ hoặc thư mục project.
from pathlib import Path
import sys

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src').is_dir()
)
sys.path.insert(0, str(project_root))
DATA_DIR = project_root / 'data'
MODEL_DIR = project_root / 'models'
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report

In [8]:
from src.models.fine_tuning import build_model, fine_tune_model

In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train_split.csv')
val_df = pd.read_csv(DATA_DIR / 'validation_split.csv')
test_df = pd.read_csv(DATA_DIR / 'test_split.csv')

# Ảnh được đổi về cùng kích thước và chuẩn hóa giống pipeline hiện tại.

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (160, 160))

    # Chuẩn hóa ảnh về [0, 1]
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def make_dataset(frame, training=False):
    paths = [str(DATA_DIR / name) for name in frame['file_name']]
    labels = frame['label'].to_numpy(dtype='float32')
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(1000)
    dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(16).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)
y_test = test_df['label'].to_numpy()

In [10]:
model = build_model()

# Pha 1: backbone đang đóng băng, chỉ học bộ phân loại mới.
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_auc', mode='max', patience=2, restore_best_weights=True
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.2, patience=1, min_lr=1e-6
)
head_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early_stop, reduce_lr],
    shuffle=False,
)

Epoch 1/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 97s 197ms/step - accuracy: 0.5550 - auc: 0.5881 - loss: 0.8630 - val_accuracy: 0.6660 - val_auc: 0.7360 - val_loss: 0.6124 - learning_rate: 1.0000e-04
Epoch 2/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 91s 208ms/step - accuracy: 0.6327 - auc: 0.6918 - loss: 0.7159 - val_accuracy: 0.7420 - val_auc: 0.8234 - val_loss: 0.5137 - learning_rate: 1.0000e-04
Epoch 3/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 70s 160ms/step - accuracy: 0.6931 - auc: 0.7609 - loss: 0.6250 - val_accuracy: 0.7720 - val_auc: 0.8643 - val_loss: 0.4641 - learning_rate: 1.0000e-04
Epoch 4/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 41s 94ms/step - accuracy: 0.7171 - auc: 0.7997 - loss: 0.5677 - val_accuracy: 0.8027 - val_auc: 0.8862 - val_loss: 0.4278 - learning_rate: 1.0000e-04
Epoch 5/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 174s 398ms/step - accuracy: 0.7433 - auc: 0.8205 - loss: 0.5379 - val_accuracy: 0.8187 - val_auc: 0.9012 - val_loss: 0.4030 - learning_rate: 1.0000e-04
Epoch 6/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 220s 

In [11]:
# Pha 2: mở một phần MobileNet và tiếp tục học từ trọng số pha 1.
# Learning rate nhỏ giúp giữ lại kiến thức pretrained đã có.
model = fine_tune_model(model)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_auc', mode='max', patience=2, restore_best_weights=True
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.2, patience=1, min_lr=1e-7
)
fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early_stop, reduce_lr],
    shuffle=False,
)

Epoch 1/15


438/438 ━━━━━━━━━━━━━━━━━━━━ 32s 71ms/step - accuracy: 0.8219 - auc: 0.9021 - loss: 0.3980 - val_accuracy: 0.8740 - val_auc: 0.9448 - val_loss: 0.3163 - learning_rate: 1.0000e-05
Epoch 2/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 29s 67ms/step - accuracy: 0.8279 - auc: 0.9071 - loss: 0.3875 - val_accuracy: 0.8747 - val_auc: 0.9450 - val_loss: 0.3158 - learning_rate: 1.0000e-05
Epoch 3/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 30s 68ms/step - accuracy: 0.8194 - auc: 0.8999 - loss: 0.3999 - val_accuracy: 0.8753 - val_auc: 0.9452 - val_loss: 0.3155 - learning_rate: 1.0000e-05
Epoch 4/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 31s 70ms/step - accuracy: 0.8231 - auc: 0.9043 - loss: 0.3922 - val_accuracy: 0.8753 - val_auc: 0.9453 - val_loss: 0.3151 - learning_rate: 1.0000e-05
Epoch 5/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 29s 67ms/step - accuracy: 0.8257 - auc: 0.9043 - loss: 0.3931 - val_accuracy: 0.8753 - val_auc: 0.9455 - val_loss: 0.3146 - learning_rate: 1.0000e-05
Epoch 6/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 58s 132ms/step - accu

In [12]:
# Chỉ dùng tập test sau khi đã hoàn tất huấn luyện.
test_loss, test_accuracy, test_auc = model.evaluate(test_ds)
print(f'Test accuracy: {test_accuracy:.4f}; AUC: {test_auc:.4f}')

probabilities = model.predict(test_ds, verbose=1).ravel()
predictions = (probabilities >= 0.5).astype('int32')
print(classification_report(
    y_test, predictions, target_names=['Real', 'AI-generated'], digits=4
))

MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save(MODEL_DIR / 'fine_tuned_mobilenet.keras')

94/94 ━━━━━━━━━━━━━━━━━━━━ 4s 46ms/step - accuracy: 0.8540 - auc: 0.9356 - loss: 0.3326
Test accuracy: 0.8540; AUC: 0.9356
94/94 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step
              precision    recall  f1-score   support

        Real     0.8480    0.8627    0.8553       750
AI-generated     0.8602    0.8453    0.8527       750

    accuracy                         0.8540      1500
   macro avg     0.8541    0.8540    0.8540      1500
weighted avg     0.8541    0.8540    0.8540      1500

